# LHFM quickstart

**Synthetic data → multimodal foundation model → bootstrap-CI AUROC, in one notebook.**

This notebook runs the full LHFM pipeline end-to-end on a small synthetic cohort. It's designed to run in either:

- **Google Colab** (free, no install needed — just hit Runtime → Run all)
- **Your laptop** after `pip install -e .` in the repo

Total runtime: ~3 minutes on Colab's free CPU, ~90 seconds on a 2020-era laptop.

What you'll see:

1. A 60-participant × 60-day synthetic cohort with medications, comorbidities, hormonal cycles, heat-waves, and wildfire smoke
2. Feature engineering into a 64-column daily panel
3. A tiny multimodal transformer (~50k params) pretrained with masked reconstruction
4. Four downstream risk heads trained with participant-aware bootstrap evaluation
5. The integrated-gradients interpretability that the dashboard surfaces
6. A subgroup-stratified fairness audit

If you only have 2 minutes, just hit *Runtime → Run all* and read the section headers.

> ⚠️ **Research prototype.** Synthetic data only. Not a medical device. Not for clinical use.
> See [`paper/ethics.md`](https://github.com/ceyhunolcan/longitudinal-health-foundation-model/blob/main/paper/ethics.md) and [`ACCEPTABLE_USE.md`](https://github.com/ceyhunolcan/longitudinal-health-foundation-model/blob/main/ACCEPTABLE_USE.md).

## 0. Setup

If you're in Colab, the cell below clones the repo and installs deps. Skip the clone if you're running locally inside the repo.

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
REPO = "longitudinal-health-foundation-model"

if IN_COLAB and not pathlib.Path(REPO).exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        f"https://github.com/ceyhunolcan/{REPO}.git",
    ], check=True)

# When running locally inside the repo, we're already in the right dir.
# When running in Colab, cd in.
if pathlib.Path(REPO).exists():
    os.chdir(REPO)

print("working dir:", os.getcwd())

In [ ]:
# Install LHFM and its dependencies. CPU torch is plenty for this notebook.
# (Colab usually has a newer torch already; we don't reinstall if so.)

try:
    import torch
    print("torch already installed:", torch.__version__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("\nInstall complete.")

In [ ]:
# Make sure `from lhfm.X import …` works whether we're in Colab or local.
src_path = str(pathlib.Path("src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lhfm
print("LHFM ready.")

## 1. Generate a synthetic cohort

60 participants × 60 days. The generator simulates:

- **Wearable** physiology (HRV, RHR, sleep duration & efficiency, daily steps, automated stress)
- **Smartphone** passive sensing (unlocks, screen time, mobility radius, location entropy)
- **EMA** self-report (mood, energy, stress on a 1-7 scale)
- **Environment** (temperature, humidity, AQI, heat index) with heat-waves, cold snaps, wildfire smoke
- **Clinical context**: medications (SSRIs, β-blockers, sleep aids), comorbidities, hormonal cycles, device-generation noise
- **Subgroup metadata** for the fairness audit (race, SES, region, device — no disparities baked in by design)
- **Informative missingness** (low-mood days are more likely to be missing)

In [ ]:
from lhfm.data.synthetic_generator import generate_synthetic_cohort

raw = generate_synthetic_cohort(n_participants=60, n_days=60, seed=7)
print(f"shape: {raw.shape}")
print(f"participants: {raw['participant_id'].nunique()}")
print(f"date range:   {raw['date'].min()} → {raw['date'].max()}")

raw.head(3)

In [ ]:
# Verify the realism checks: within-person mood-sleep correlation,
# β-blocker HRV effect.
corrs = []
for pid, g in raw.groupby("participant_id"):
    g = g.dropna(subset=["survey_mood", "sleep_duration"])
    if len(g) >= 15:
        c = g[["survey_mood", "sleep_duration"]].corr().iloc[0, 1]
        if not np.isnan(c):
            corrs.append(c)
print(f"within-person mood-sleep r: mean={np.mean(corrs):.3f}  (real EMA studies: 0.25-0.35)")

by_bb = raw.groupby("participant_id").agg(
    on_bb=("on_beta_blocker", "first"),
    mean_hrv=("hrv_rmssd", "mean"),
    mean_rhr=("resting_hr", "mean"),
)
bb = by_bb[by_bb.on_bb == 1]
nb = by_bb[by_bb.on_bb == 0]
if len(bb) >= 3:
    print(f"β-blocker effect: HRV {bb.mean_hrv.mean() - nb.mean_hrv.mean():+.1f} ms, "
          f"RHR {bb.mean_rhr.mean() - nb.mean_rhr.mean():+.1f} bpm  "
          f"(meta-analytic target: +10 ms / -10 bpm)")

## 2. Feature engineering

Five feature modules turn the raw long-form frame into a 64-column daily panel:

- **wearable**: HRV / RHR deviation from personal baseline, stress burden
- **smartphone**: screen-time z-scores, mobility surprise
- **climate**: heat-index lags, AQI rolling means
- **missingness**: per-modality missingness rates
- **baseline**: age z-score, sex/chronotype one-hots

All within-person standardization uses *expanding* (past-only) windows — no temporal leakage.

In [ ]:
from lhfm.features import build_full_feature_table

feat = build_full_feature_table(raw, impute=True, add_targets=True)
target_cols = [c for c in feat.columns if c.startswith("target_")]
print(f"feature table: {feat.shape}")
print("\ntarget balance:")
for t in target_cols:
    vals = feat[t].dropna()
    print(f"  {t:30s} positive_rate={vals.mean():.3f}  n={len(vals)}")

## 3. Train the foundation model

Two-stage protocol:

1. **Self-supervised pretrain** — masked reconstruction over multimodal windows. The encoder learns to fill in masked features from context, building a representation that's useful before we see any labels.
2. **Downstream fine-tune** — four binary heads on the 14-day windows: low mood, high stress, sleep disruption, climate vulnerability.

For this notebook we use a tiny model (d=64, 2 layers, ~50k params) so it trains in ~60 seconds on CPU. In the full pipeline the same architecture scales to d=256, 6 layers.

In [ ]:
import torch
from lhfm.data.preprocessing import build_windows, train_val_test_split_by_participant
from lhfm.features.baseline_features import compute_baseline_features, fit_baseline_reference_stats
from lhfm.models.encoder import MultimodalLongitudinalEncoder
from lhfm.training.dataset import LongitudinalWindowDataset
from lhfm.training.train_downstream import train_downstream
from lhfm.training.train_ssl import pretrain_ssl
from lhfm.utils.config import set_global_seed

set_global_seed(7)

splits = train_val_test_split_by_participant(
    feat, val_fraction=0.15, test_fraction=0.20, seed=7,
)
# Re-standardize age against the train split only.
ref_stats = fit_baseline_reference_stats(splits["train"])
for k in ("train", "val", "test"):
    splits[k] = compute_baseline_features(
        splits[k],
        age_ref_mean=ref_stats["age_ref_mean"],
        age_ref_std=ref_stats["age_ref_std"],
    )

drop = {"participant_id", "date", "sex", "chronotype", "race_ethnicity",
        "ses_proxy", "region", "device_gen", "cycle_phase"} | set(target_cols)
feature_cols = [c for c in feat.columns if c not in drop and feat[c].dtype.kind in "fi"]
print(f"using {len(feature_cols)} numeric features")

def _build(split_df):
    X, _, pids, end_dates = build_windows(
        split_df, feature_cols=feature_cols,
        target_col=target_cols[0], window_days=14, stride=1, target_mode="next_day",
    )
    if X.shape[0] == 0:
        return None
    target_dates = pd.to_datetime(end_dates) + pd.Timedelta(days=1)
    long = split_df[["participant_id", "date", *target_cols]].copy()
    long["date"] = pd.to_datetime(long["date"])
    long = long.set_index(["participant_id", "date"])
    Y = np.full((X.shape[0], len(target_cols)), np.nan, dtype=np.float32)
    for j, key in enumerate(zip(pids.tolist(), target_dates)):
        try:
            row = long.loc[key]
            for i, col in enumerate(target_cols):
                v = row[col]
                Y[j, i] = float(v) if not pd.isna(v) else np.nan
        except KeyError:
            pass
    return X, Y, pids

Xtr, Ytr, pids_tr = _build(splits["train"])
Xva, Yva, pids_va = _build(splits["val"])
Xte, Yte, pids_te = _build(splits["test"])

all_pids = sorted(feat["participant_id"].unique())
pid_to_idx = {p: i for i, p in enumerate(all_pids)}
idx_tr = np.array([pid_to_idx[p] for p in pids_tr], dtype=np.int64)
idx_va = np.array([pid_to_idx[p] for p in pids_va], dtype=np.int64)
idx_te = np.array([pid_to_idx[p] for p in pids_te], dtype=np.int64)

modality_slices = {"all": (0, len(feature_cols))}
train_ds = LongitudinalWindowDataset(Xtr, Ytr, modality_slices, participant_idx=idx_tr)
val_ds = LongitudinalWindowDataset(Xva, Yva, modality_slices, participant_idx=idx_va)
test_ds = LongitudinalWindowDataset(Xte, Yte, modality_slices, participant_idx=idx_te)
print(f"windows: train={Xtr.shape[0]}, val={Xva.shape[0]}, test={Xte.shape[0]}")

In [ ]:
encoder = MultimodalLongitudinalEncoder(
    modality_dims={"all": len(feature_cols)},
    d_model=64, n_heads=4, n_layers=2, max_seq_len=14,
    n_participants=len(all_pids),
)
n_params = sum(p.numel() for p in encoder.parameters())
print(f"encoder: {n_params:,} parameters")

print("\nSSL pretrain (3 epochs)...")
pretrain_ssl(
    encoder=encoder,
    reconstruction_target_modality="all",
    train_dataset=train_ds, val_dataset=val_ds,
    epochs=3, batch_size=32, lr=3e-4, weight_decay=0.0,
    device="cpu", checkpoint_path=None,
)

print("\nDownstream fine-tune (5 epochs)...")
task_names = [c.replace("target_", "") for c in target_cols]
state = train_downstream(
    encoder=encoder, task_names=task_names,
    train_dataset=train_ds, val_dataset=val_ds,
    epochs=5, batch_size=32, lr=5e-4, weight_decay=1e-5,
    device="cpu", freeze_encoder=False, checkpoint_path=None,
    early_stopping_patience=10,
)

## 4. Evaluate with participant-clustered bootstrap CIs

Standard bootstrap resamples rows. That's wrong for longitudinal data: 14-day windows from the same participant are correlated, so resampling rows underestimates uncertainty. The participant-cluster bootstrap resamples *people*, then takes all their windows together.

Effect: CIs are typically 3-5× wider than the row-level bootstrap would give you. That's the honest width.

In [ ]:
from lhfm.training.evaluate import evaluate_downstream

results = evaluate_downstream(
    state.model, test_ds, task_names=task_names,
    device="cpu", batch_size=32, bootstrap_resamples=300,
)

print(f"{'task':<22} {'AUROC':<8} {'95% CI':<20} {'AUPRC':<8} {'ECE':<6} {'n_test':<6}")
print("-" * 78)
for task, r in results.items():
    ci = r.get("auroc_ci", (float("nan"), float("nan")))
    print(f"  {task:<20} {r['auroc']:.3f}    [{ci[0]:.3f}, {ci[1]:.3f}]      "
          f"{r['auprc']:.3f}    {r['ece']:.3f}  {r['n_total']}")

## 5. Interpretability: which features drove a prediction?

Integrated gradients (Sundararajan, Taly & Yan 2017) attributes the prediction to each (modality × feature × day) input. Positive scores push toward "risk elevated"; negative push toward "risk low".

We pick the highest-risk test window and ask: *which features got it there?*

In [ ]:
from lhfm.interpretability import attribute, aggregate_to_feature_table

# Find the test window with the highest predicted low-mood risk.
state.model.eval()
with torch.no_grad():
    from torch.utils.data import DataLoader
    from lhfm.training.dataset import collate_windows
    loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_windows)
    risks = []
    for batch in loader:
        logits = state.model(batch["modalities"], batch["mask"], batch["participant_idx"])
        risks.append(torch.sigmoid(logits["low_mood"]).cpu().numpy())
    risks = np.concatenate(risks)

highest_idx = int(np.argmax(risks))
print(f"selected test window {highest_idx} (predicted low_mood risk: {risks[highest_idx]:.3f})")

# Get the window's input data.
window = test_ds[highest_idx]
modalities_in = {k: v.unsqueeze(0) for k, v in window["modalities"].items()}

attrs = attribute(
    model=state.model,
    modalities=modalities_in,
    task="low_mood",
    mask=window["mask"].unsqueeze(0),
    participant_idx=window["participant_idx"].unsqueeze(0),
    n_steps=32,
)

feat_table = aggregate_to_feature_table(
    attrs,
    feature_names={"all": feature_cols},
)
print("\nTop 10 features by absolute attribution:")
feat_table.head(10)

In [ ]:
# Visualise: bar chart of top features by attribution.
top = feat_table.head(12).copy()
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#c2185b" if v > 0 else "#1976d2" for v in top["attribution"]]
ax.barh(top["feature"][::-1], top["attribution"][::-1], color=colors[::-1])
ax.axvline(0, color="#444", linewidth=0.8)
ax.set_xlabel("integrated-gradient attribution")
ax.set_title(f"What drove this low_mood prediction? (risk = {risks[highest_idx]:.2f})")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Subgroup-stratified fairness audit

The synthetic generator draws subgroup attributes (race, SES, region, device generation) independently of outcomes, so a well-trained model should pass with low gaps. On real data this audit is where real disparities surface.

We compute per-subgroup AUROC with participant-clustered bootstrap CIs, plus the equalized-odds gap (max FPR-gap + max FNR-gap) per axis.

In [ ]:
from lhfm.utils.fairness import run_fairness_audit, check_fairness_thresholds

# Build per-window metadata aligned to the test set.
test_sub = splits["test"][["participant_id", "date",
                            "sex", "age", "race_ethnicity",
                            "ses_proxy", "region", "device_gen"]].copy()
test_sub["date"] = pd.to_datetime(test_sub["date"])
_, _, pids_te2, end_dates_te2 = build_windows(
    splits["test"], feature_cols=feature_cols,
    target_col=target_cols[0], window_days=14, stride=1, target_mode="next_day",
)
end_dates_pd = pd.to_datetime(end_dates_te2)
lookup = test_sub.set_index(["participant_id", "date"])
rows_meta = []
for pid_i, edate in zip(pids_te2.tolist(), end_dates_pd):
    try:
        rows_meta.append(lookup.loc[(pid_i, edate)].to_dict())
    except KeyError:
        rows_meta.append({})
md = pd.DataFrame(rows_meta)

# Get y_true / y_prob for the low_mood task.
low_mood_idx = task_names.index("low_mood")
yt = Yte[:, low_mood_idx]
valid = ~np.isnan(yt)

audit = run_fairness_audit(
    yt[valid], risks[valid], md.iloc[valid].reset_index(drop=True),
    min_subgroup_n=20, bootstrap_resamples=200,
    groups=idx_te[valid],
)

rows = pd.DataFrame(audit["per_subgroup"])
print("Per-subgroup AUROC for low_mood prediction:\n")
rows[["axis", "level", "n", "auroc", "auprc", "fpr", "fnr"]].round(3)

In [ ]:
ok, violations = check_fairness_thresholds(audit, max_auroc_gap=0.10, max_eo_violation=0.20)
if ok:
    print("✓ No fairness threshold violations on this synthetic cohort.")
    print("  (Expected — the generator draws subgroups independently of outcomes.)")
    print("  Real cohorts will likely surface gaps. That's where the audit becomes interesting.")
else:
    print("✗ Violations detected:")
    for v in violations:
        print(f"  - {v}")

## 7. Where to next

You've just run the full LHFM pipeline end-to-end. The tiny model + tiny cohort means the AUROC numbers aren't yet competitive — but every methodological piece is in place. To scale this up:

- **Bigger synthetic run**: `make data` (250 participants × 90 days) then `make train`
- **Real data**: see `docs/adapters/lifesnaps.md` (LifeSnaps, Kaggle, instant) or `docs/adapters/globem.md` (GLOBEM, PhysioNet, ~10 days credentialing)
- **Interactive dashboard**: `streamlit run src/lhfm/dashboard/app.py` — per-participant timeline with per-day attribution
- **Climate-regime generalization**: `make climate-holdout` — hold out heat-wave days, evaluate on them
- **Scale ablation**: `make scale-ablation` — AUROC vs. pretraining cohort size, the canonical foundation-model curve

Full repo: <https://github.com/ceyhunolcan/longitudinal-health-foundation-model>

Methodology paper draft: [`paper/methods.md`](https://github.com/ceyhunolcan/longitudinal-health-foundation-model/blob/main/paper/methods.md)